# 0 — model-compatibility check (new model / tokenizer)

Before running the clock localization joblist (`job_list/clock_olmo32b_deps.json`) on a **new model**
(e.g. `allenai/Olmo-3.1-32B-Instruct`), validate every *hardcoded assumption* the pipeline bakes in —
most were written against `Qwen2.5-Coder-*` and may not hold for a different tokenizer/architecture.

Checks (each prints ✅/❌; nothing aborts — a summary is printed at the end):

| § | assumption | scripts affected |
|---|---|---|
| A | model whitelisted + config fully parsed (`n_layer`, formats, o_proj/mlp modules) | **all** |
| B | `o_proj.in_features == num_attention_heads · head_dim` (per-head split of the o_proj input) | `train_completion_mask(_gaze)`, `train_head_mask`, `GazeScorer` |
| C | `apply_chat_template` opens the assistant turn with `<code>` and injects **no** thinking block | `build_condition_data`, both mask trainers |
| D | **the pivot**: augmented prompt ⇒ token `[-2]` = `"    \n"`, token `[-1]` = `"   "` | `train_completion_mask(_gaze)` (patch `-2`, read `-1`) |
| E | `naming="letters"` verdicts (`" a"…" f"`) are single, distinct tokens | both mask trainers (logit readout) |
| F | gaze Q·K assumes **no QK-norm** (and rope only if `--gaze-rope`) | `train_completion_mask_gaze` (`GazeScorer`) |
| G | the full pair pipeline + goal-head prompt path actually build on this model | all |

In [1]:
import sys, os
for _p in ["../..", "..", "."]:
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, os.path.abspath(_p)); break

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from src.models import ModelandTokenizer
from src.functional import get_module_nnsight

# ── model under test ──
MODEL = "allenai/Olmo-3.1-32B-Instruct"
# ──────────────────────

RESULTS = []
def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail))
    print(f"{'✅' if ok else '❌'} {name}" + (f"  — {detail}" if detail else ""))
def note(msg):
    print(f"ℹ️  {msg}")

In [2]:
mt = ModelandTokenizer(
    model_key=MODEL, dtype=torch.bfloat16, device_map="auto", attn_implementation="eager",
)
print("loaded:", mt.name)

allenai/Olmo-3.1-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

loaded: allenai/Olmo-3.1-32B-Instruct


## §A — whitelist + config parse
Every script does `ModelandTokenizer(model_key=...)`. If the architecture isn't recognized, the module-name formats come back `None` and every downstream `.format(l)` breaks.

In [3]:
from src.models import is_olmo_variant

check("is_olmo_variant (or otherwise recognized)", is_olmo_variant(mt) or all(
    getattr(mt, f) is not None for f in ["attn_module_name_format", "mlp_module_name_format"]),
    f"olmo={is_olmo_variant(mt)}")

for f in ["n_layer", "n_embd", "layer_name_format", "attn_module_name_format", "mlp_module_name_format"]:
    v = getattr(mt, f, None)
    check(f"config.{f} set", v is not None, repr(v))

# modules resolve for layer 0
ok_mod = True; detail = ""
try:
    o0 = get_module_nnsight(mt._model, mt.attn_module_name_format.format(0) + ".o_proj")
    m0 = get_module_nnsight(mt._model, mt.mlp_module_name_format.format(0))
    detail = f"o_proj={type(o0).__name__}, mlp={type(m0).__name__}"
except Exception as e:
    ok_mod = False; detail = f"{type(e).__name__}: {e}"
check("layer-0 o_proj + mlp modules resolve", ok_mod, detail)
print(f"\nn_layer={mt.n_layer}  n_embd={mt.n_embd}  attn_fmt={mt.attn_module_name_format}")

✅ is_olmo_variant (or otherwise recognized)  — olmo=True
✅ config.n_layer set  — 64
✅ config.n_embd set  — 5120
✅ config.layer_name_format set  — 'model.layers.{}'
✅ config.attn_module_name_format set  — 'model.layers.{}.self_attn'
✅ config.mlp_module_name_format set  — 'model.layers.{}.mlp'
✅ layer-0 o_proj + mlp modules resolve  — o_proj=Linear, mlp=Olmo3MLP

n_layer=64  n_embd=5120  attn_fmt=model.layers.{}.self_attn


## §B — head geometry
`completion_patch_hooks` / `_extract_src_hv` / `GazeScorer` split the **o_proj input** into `num_attention_heads` chunks of `head_dim`. This requires `o_proj.in_features == num_attention_heads · head_dim`. (GQA is fine — the o_proj input is the full concatenated query-head output, independent of `num_key_value_heads`.)

In [4]:
cfg = mt.model.config
nH = cfg.num_attention_heads
hd = getattr(cfg, "head_dim", mt.n_embd // nH)
nKV = getattr(cfg, "num_key_value_heads", nH)

o0 = get_module_nnsight(mt._model, mt.attn_module_name_format.format(0) + ".o_proj")
in_features = o0.weight.shape[1]   # (out=hidden, in=nH*head_dim)

check("o_proj.in_features == num_attention_heads * head_dim", in_features == nH * hd,
      f"in_features={in_features}, nH*hd={nH*hd} (nH={nH}, head_dim={hd})")
if nH * hd != mt.n_embd:
    note(f"nH*head_dim ({nH*hd}) != hidden_size ({mt.n_embd}) — fine as long as the check above passed, "
         f"but any code that assumes o_proj input width == n_embd would be wrong.")
note(f"num_key_value_heads={nKV} (GQA group={nH // max(nKV,1)}) — not used for o_proj splitting.")

✅ o_proj.in_features == num_attention_heads * head_dim  — in_features=5120, nH*hd=5120 (nH=40, head_dim=128)
ℹ️  num_key_value_heads=8 (GQA group=5) — not used for o_proj splitting.


## §C — chat template
`apply_chat_template` builds a 3-turn chat and leaves the **assistant turn open** right after `<code>\n{stub}` (via `continue_final_message=True`), passing `enable_thinking=False`. If Olmo-3 injects a reasoning/thinking scaffold or closes the turn, the generation continues in the wrong place and the pivot logic is meaningless.

In [5]:
from src.goals.functional import apply_chat_template, supports_chat_template

check("supports_chat_template(mt)", supports_chat_template(mt), f"name={mt.name!r}")

stub = "def check_conditions(n: int):\n    a = n > 0\n    "
ok_ct = True; templ = ""
try:
    templ = apply_chat_template(mt=mt, user_query="Implement the checks.", incomplete_code=stub)
except Exception as e:
    ok_ct = False; templ = f"{type(e).__name__}: {e}"
check("apply_chat_template runs", ok_ct, "")
if ok_ct:
    check("assistant turn contains '<code>'", "<code>" in templ)
    check("prompt ends at our stub (turn left open)", templ.rstrip().endswith(stub.rstrip()),
          f"...{templ[-60:]!r}")
    check("no injected <think> scaffold", "<think>" not in templ.lower() and "reasoning" not in templ[-200:].lower())
    print("\n----- tail of templated prompt -----\n" + templ[-400:])

✅ supports_chat_template(mt)  — name='allenai/Olmo-3.1-32B-Instruct'
✅ apply_chat_template runs
✅ assistant turn contains '<code>'
✅ prompt ends at our stub (turn left open)  — ...'tant\n<code>\ndef check_conditions(n: int):\n    a = n > 0\n    '
✅ no injected <think> scaffold

----- tail of templated prompt -----
<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Implement the checks.<|im_end|>
<|im_start|>assistant
<code>
def check_conditions(n: int):
    a = n > 0
    


## §D — the pivot (the core assumption) ⭐
The mask trainers augment each prompt with `replace_in_code_newline_with_tabnewline(prompt) + "   "`, then **patch at token `-2`** and **read the verdict at token `-1`**. This bakes in: `[-2]` = the `"    \n"` pivot (4 spaces + newline), `[-1]` = the trailing `"   "`. On a different tokenizer these may fragment/merge, shifting the pivot.

In [6]:
from src.goals.completion_pairs import build_prompt
from src.tokens import prepare_input
import random

# ml (staying / incomplete) prompt — its target line ends at the pivot
ml_prompt, _ = build_prompt(mt, random.Random(0), target_slot=1, n_conditions=4, mode="chat", style="multiline")
sl_prompt, _ = build_prompt(mt, random.Random(0), target_slot=1, n_conditions=4, mode="chat", style="singleline")

ml_ids = prepare_input(ml_prompt, tokenizer=mt, padding_side="left")["input_ids"][0]
sl_ids = prepare_input(sl_prompt, tokenizer=mt, padding_side="left")["input_ids"][0]

print("ml prompt tail tokens:")
for k in range(-6, 0):
    print(f"  [{k}] {mt.tokenizer.decode([ml_ids[k].item()])!r}")

tok_last  = mt.tokenizer.decode([ml_ids[-1].item()])
tok_pivot = mt.tokenizer.decode([ml_ids[-2].item()])

check("token [-1] is the trailing '   ' (pure spaces)", set(tok_last) <= {" "} and len(tok_last) >= 1, repr(tok_last))
check("token [-2] is the pivot '    \\n' (whitespace incl newline)",
      ("\n" in tok_pivot) and (tok_pivot.replace(" ", "").replace("\n", "") == ""), repr(tok_pivot))
check("pivot token [-2] aligns across ml/sl styles (validate_pair assumption)",
      ml_ids[-2].item() == sl_ids[-2].item(),
      f"ml={mt.tokenizer.decode([ml_ids[-2].item()])!r} sl={mt.tokenizer.decode([sl_ids[-2].item()])!r}")

ml prompt tail tokens:
  [-6] ' str'
  [-5] '(abs'
  [-4] '(n'
  [-3] ')))'
  [-2] '    \n'
  [-1] '   '
✅ token [-1] is the trailing '   ' (pure spaces)  — '   '
✅ token [-2] is the pivot '    \n' (whitespace incl newline)  — '    \n'
✅ pivot token [-2] aligns across ml/sl styles (validate_pair assumption)  — ml='    \n' sl='    \n'


## §E — single-token letter verdicts
With `naming="letters"` the verdict is a variable name (`a`,`b`,`c`,…). The logit readout `logp[target_ans] - logp[dst_ans]` uses a **single** next-token id, so `" a"…" f"` should each be one token and mutually distinct.

In [7]:
letters = "abcdef"
ids_per = {ch: mt.tokenizer(" " + ch, add_special_tokens=False).input_ids for ch in letters}
all_single = all(len(v) == 1 for v in ids_per.values())
firsts = [v[0] for v in ids_per.values()]
all_distinct = len(set(firsts)) == len(firsts)

check("' a'..' f' each tokenize to a single token", all_single,
      {ch: len(v) for ch, v in ids_per.items()})
check("first tokens are mutually distinct", all_distinct, firsts)

✅ ' a'..' f' each tokenize to a single token  — {'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 1, 'f': 1}
✅ first tokens are mutually distinct  — [264, 293, 272, 294, 384, 282]


## §F — gaze objective (`GazeScorer`) caveats
`score_condition_heads` uses the model's **real** attention weights (correct anywhere). But `GazeScorer` (the *training objective* in `train_completion_mask_gaze`) recomputes Q·K **manually** from `q_proj`/`k_proj` and **does not apply QK-norm**. Qwen2.5 has no QK-norm; **Olmo2/3 do** → the manual gaze affinity won't match real attention. Rope is only used with `--gaze-rope` (off in our joblist).

In [8]:
attn0 = get_module_nnsight(mt._model, mt.attn_module_name_format.format(0))
has_qnorm = hasattr(attn0, "q_norm") and getattr(attn0, "q_norm", None) is not None
has_knorm = hasattr(attn0, "k_norm") and getattr(attn0, "k_norm", None) is not None
has_qkproj = hasattr(attn0, "q_proj") and hasattr(attn0, "k_proj")

check("attn has q_proj/k_proj (GazeScorer prerequisite)", has_qkproj)
# This is a WARNING check: pass = safe for GazeScorer as-written.
check("model has NO QK-norm (GazeScorer omits it)", not (has_qnorm or has_knorm),
      f"q_norm={has_qnorm}, k_norm={has_knorm}")
if has_qnorm or has_knorm:
    note("→ GazeScorer.__proj applies q_proj/k_proj WITHOUT q_norm/k_norm. On this model the gaze "
         "objective is miscalibrated. Either patch GazeScorer to apply q_norm/k_norm, or prefer the "
         "logit-objective decision heads (train_completion_mask).")

has_rotary = hasattr(mt._model.model, "rotary_emb")
check("model.model.rotary_emb present (only needed for --gaze-rope)", has_rotary)

✅ attn has q_proj/k_proj (GazeScorer prerequisite)
❌ model has NO QK-norm (GazeScorer omits it)  — q_norm=True, k_norm=True
ℹ️  → GazeScorer.__proj applies q_proj/k_proj WITHOUT q_norm/k_norm. On this model the gaze objective is miscalibrated. Either patch GazeScorer to apply q_norm/k_norm, or prefer the logit-objective decision heads (train_completion_mask).
✅ model.model.rotary_emb present (only needed for --gaze-rope)


## §G — end-to-end pipeline build
Actually build a few completion pairs (the exact `build_completion_pairs` the trainers call) and sanity-check the goal-head prompt path (`head_loc_pivot`). Reports how many pairs survive the alignment + contrast gates on this model.

In [9]:
from src.goals.completion_pairs import build_completion_pairs

pairs = build_completion_pairs(
    mt, n_pairs=4, target_slots=(1, 2, 3), direction="both", same=False,
    max_conditions=6, mode="chat", seed=0, patch_tokens=(-2,), log=lambda *a, **k: None,
)
check("build_completion_pairs yields validated pairs", len(pairs) > 0, f"{len(pairs)}/4 built")
if pairs:
    p = pairs[0]
    note(f"pair0: dst clean={mt.tokenizer.decode([p.dest.ml_ans])!r} vs "
         f"moved-on={mt.tokenizer.decode([p.dest.sl_ans])!r}")

# goal-head prompt path (mine_pivot_instances / train_head_mask use head_loc_pivot builders)
try:
    from src.goals.head_loc_data import HeadLocDataset
    from src.goals.head_loc_pivot import build_python_prompt, build_cpp_prompt
    ds = HeadLocDataset.load(model_name=MODEL)
    s0 = ds.samples[0] if hasattr(ds, "samples") else list(ds.predicate_samples.values())[0]
    _py = build_python_prompt(mt, s0, "", True)
    _cpp = build_cpp_prompt(mt, s0, True)
    check("head_loc_pivot python/cpp prompts build (goal-head path)", bool(_py) and bool(_cpp),
          f"py_len={len(_py)}, cpp_len={len(_cpp)}")
except Exception as e:
    RESULTS.append(("head_loc_pivot prompts build (goal-head path)", None, f"SKIP: {type(e).__name__}: {e}"))
    print(f"⏭️  head_loc_pivot goal-head path — SKIP: {type(e).__name__}: {e}")

✅ build_completion_pairs yields validated pairs  — 4/4 built
ℹ️  pair0: dst clean=' b' vs moved-on=' c'
✅ head_loc_pivot python/cpp prompts build (goal-head path)  — py_len=408, cpp_len=388


## Summary

In [10]:
print("=" * 70)
_p = sum(1 for _, ok, _ in RESULTS if ok is True)
_f = [(n, d) for n, ok, d in RESULTS if ok is False]
_s = [(n, d) for n, ok, d in RESULTS if ok is None]
print(f"{_p} passed · {len(_f)} FAILED · {len(_s)} skipped   (of {len(RESULTS)} checks)")
if _f:
    print("\nFAILURES (must fix before running the joblist):")
    for n, d in _f:
        print(f"  ❌ {n}" + (f"  — {d}" if d else ""))
if _s:
    print("\nskipped:")
    for n, d in _s:
        print(f"  ⏭️  {n}  — {d}")
print("=" * 70)

22 passed · 1 FAILED · 0 skipped   (of 23 checks)

FAILURES (must fix before running the joblist):
  ❌ model has NO QK-norm (GazeScorer omits it)  — q_norm=True, k_norm=True


In [13]:
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape
import logging
from src.utils import logging_utils

console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)

In [14]:
from src.goals.completion_pairs import build_completion_pairs, LEG_STYLES
from src.goals.head_loc_attn import get_o_proj_inputs_batch
from src.tokens import prepare_input

# ── config for the mean signal ──
HP_DIRECTION   = "sl->ml"          # "sl->ml" = MOVE-ON mean (push a stay dest to move on)
                                   # "ml->sl" = STAY    mean (push a move-on dest to stay)
N_MEAN_SRC     = 200
MEAN_SRC_BATCH = 16
MEAN_SEED      = 1234

src_style = "sl" if HP_DIRECTION == "sl->ml" else "ml"
sig_name  = "move-on" if src_style == "sl" else "stay"

ss = LEG_STYLES[HP_DIRECTION][
    "src_style"
]  # source donates this style (== dest's target)
ds = LEG_STYLES[HP_DIRECTION]["dst_style"]  # dest runs (clean) this style

print(f"{ss=}, {ds=}")

ss='sl', ds='ml'


In [17]:
from src.goals.multiline_prompts import (
    build_example,
    replace_in_code_newline_with_tabnewline,
)
from src.functional import generate_with_patch

#####################################################################
NAMING = "letters"  # "letters" | "natural" | "indexed"
N_GEN = 32
#####################################################################

seed = random.Random(1234)
souce_example = build_example(
    mt=mt,
    rng=seed,
    target_slot=1,
    n_conditions=4,
    mode="chat",
    # style="singleline" if ss=="sl" else "multiline",
    style="multiline" if ss == "sl" else "singleline",
    naming=NAMING,
)

src_prompt = souce_example["full_prompt"]

prefix=""

src_prompt = src_prompt + prefix
src_prompt = replace_in_code_newline_with_tabnewline(src_prompt) + "   "

source_gen = generate_with_patch(
    mt,
    inputs=src_prompt,
    n_gen_per_prompt=1,
    max_new_tokens=N_GEN,
    do_sample=False,
    remove_prefix=True,
)[0]
rich_print(
    f"[white]{escape(src_prompt)}[/white][medium_orchid1]{escape(source_gen)}[/medium_orchid1]"
)

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. the product of n's digits is greater than 0
2. n is a perfect square
3. n is even
4. n is divisible by 4

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code>    
def check_conditions(n: int):    
    a = 1    
    for d in str(abs(n)):    
        a *= int(d)    
    a = a > 0    
    b = int(abs(n) ** 0.5)    
    b = b * b == n    
    c = n % 2 == 0    
    d = n % 4 == 0    
    print